# 01 — Data Exploration

Before touching any ML, we need to understand the raw signal.

**This notebook:**
- Loads REDD House 1 low-frequency data
- Plots aggregate mains over time
- Plots individual appliance traces
- Checks data quality: gaps, nulls, sampling regularity

**Why this matters:** For signal-based problems you must look before you compute. Feature choices depend on what the signal actually looks like.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from utils.signal_utils import load_refit_house

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120

## Load the data

REDD House 1 has two mains channels (two electrical phases) plus ~20 appliance sub-meters.
We sum the two phases to get total aggregate power — that is what a single utility meter would see.

In [ ]:
# Path to your downloaded REFIT CSV file
# Path to your downloaded REFIT CSV file
HOUSE_FILE = '../data/raw/CLEAN_House1.csv'
df, labels = load_refit_house(HOUSE_FILE)

print('Shape:', df.shape)
print('Date range:', df.index[0], 'to', df.index[-1])
print('Appliances:', [c for c in df.columns if c != 'mains'])
df.head()
df, labels = load_refit_house(HOUSE_FILE)

print('Shape:', df.shape)
print('Date range:', df.index[0], 'to', df.index[-1])
print('Appliances:', [c for c in df.columns if c != 'mains'])
df.head()

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (1582578337.py, line 2)

In [ ]:
# check for gaps and null values
print('Null counts per channel:')
print(df.isnull().sum())

time_diffs = pd.Series(df.index).diff().dropna()
print(f'\nSampling: median={time_diffs.median()}, max gap={time_diffs.max()}')
# large max gap = recording interruptions — normal in REDD

## Aggregate mains over time

This is what the electricity meter sees — everything summed.
Each step or spike is an appliance switching.

In [ ]:
sample = df['mains'].iloc[:3*86400]  # 3 days at 1Hz

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample.index, sample.values, color='steelblue', linewidth=0.6, alpha=0.85)
ax.set_ylabel('Power (Watts)', fontsize=11)
ax.set_title('Aggregate Mains Power — 3 Days', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.savefig('../outputs/01_mains_overview.png', dpi=150)
plt.show()

## Individual appliance traces

Now look at each appliance in isolation. This is where EE intuition matters:
- **Refrigerator**: cycles on/off periodically — compressor behavior
- **Microwave**: short rectangular high-power pulses
- **Washer/Dryer**: long duration, power varies as wash cycle changes
- **Lighting**: clean step change, very stable steady state

These visual differences are exactly what the ML model will learn to distinguish.

In [ ]:
appliances_to_plot = [c for c in df.columns if c != 'mains'][:5]
colors = ['darkorange', 'mediumseagreen', 'crimson', 'mediumpurple', 'steelblue']

fig, axes = plt.subplots(len(appliances_to_plot), 1, figsize=(14, 3*len(appliances_to_plot)), sharex=True)

for ax, col, color in zip(axes, appliances_to_plot, colors):
    data = df[col].iloc[:86400]
    ax.plot(data.index, data.values, color=color, linewidth=0.7)
    ax.set_ylabel('Watts', fontsize=9)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

axes[-1].set_xlabel('Time of Day', fontsize=11)
fig.suptitle('Individual Appliance Traces — 24 Hours', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/01_appliance_traces.png', dpi=150)
plt.show()

## Energy contribution by appliance

How much does each load contribute to total consumption?
Low-power appliances are harder to identify in the aggregate — they get drowned out.

In [ ]:
appliance_cols = [c for c in df.columns if c != 'mains']
energy = df[appliance_cols].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
energy.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Total Energy (Watt-seconds)', fontsize=11)
ax.set_title('Energy Contribution by Appliance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/01_energy_contribution.png', dpi=150)
plt.show()

In [ ]:
# save 7-day sample for downstream notebooks
df_sample = df.iloc[:7*86400]
df_sample.to_csv('../data/processed/house1_7days.csv')
print(f'Saved: {df_sample.shape}')